# Reproducing Rouse & Schieber (2015) from dandiset 000941

**Paper.** Rouse AG, Schieber MH (2015). Spatiotemporal distribution of location and object effects
in reach-to-grasp kinematics. *J Neurophysiol* 114:3268-3282.
[doi:10.1152/jn.00686.2015](https://doi.org/10.1152/jn.00686.2015). The claims file
(`claims/000941/10.1152-jn.00686.2015.yaml`) was written from the version of record (Methods,
Results, legends, Tables 1 and 2; the paper has no supplement).

**Dandiset.** [DANDI:000941](https://dandiarchive.org/dandiset/000941/0.241029.1405), "FALCON
Benchmark M1-A", version **0.241029.1405**. The paper cites no version of the dataset (it predates
it), and this is the only published version, so it is the one pinned. The draft has the same 11
assets and was last modified within a second of the publication of this version.

**What the dandiset holds.** Monkey L only: trial tables, 16 channels of preprocessed EMG at 50 Hz
and multiunit spike times, for four sessions (one of them the paper's session L20120928) split for
the FALCON benchmark, and three later sessions with ten trials each. It holds no marker positions
and no joint angles, the data every kinematic claim of the paper is about, and no data of monkeys X
and Y. Only the trial table can be used here.

| Claim | Tested here |
|---|---|
| table2-trial-counts | counts shown; blocked, data missing (the dataset holds a part of the session's successful trials and no failed trials) |
| table2-movement-time | tested on monkey L (X and Y are not in the dataset) |
| fig2-trial-count-coax-45 | count shown; blocked, data missing (a part of the session's trials) |
| fig2-joint-patterns, fig2-hand-opening-two-phases | blocked, data missing (no marker positions or joint angles) |
| fig4-armps-time-course, the eleven Fig. 5 to Fig. 7 claims | blocked, data missing (no joint angles) |
| fig8a-lda-location, fig8b-lda-object, fig8-hand-shape-invariant-to-location | blocked, data missing (no joint angles) |
| fig9-hand-configuration-location, fig10-forearm-trajectory-object, elbow-point-trajectory-object | blocked, data missing (no marker positions) |

In [ ]:
import time
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import rouse

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

started = time.time()
assets = rouse.assets()
print(f"dandiset {rouse.DANDISET_ID} version {rouse.DANDISET_VERSION}: {len(assets)} assets, "
      f"{assets['size'].sum() / 1e6:.1f} MB")
print(assets[["path", "size", "split", "session"]].to_string())

files = {(r.split, r.session): rouse.open_file(r.url) for r in assets.itertuples()}
trials = {k: rouse.read_trials(f) for k, f in files.items()}
key = ("held-in-calib", rouse.PAPER_SESSION)
t = trials[key]
print(f"\nloaded the trial tables of every file in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
f = files[key]
print("Everything the file of the paper's session holds (held-in-calib, 20120928):")
def show(name, obj):
    import h5py
    if name.startswith("specifications"):
        return
    if isinstance(obj, h5py.Dataset):
        desc = obj.attrs.get("description", "")
        unit = obj.attrs.get("unit", "")
        print(f"  {name} {obj.shape} {obj.dtype}" + (f" unit={unit}" if unit else "") + (f" -- {desc}" if desc else ""))
f.visititems(show)
summary = pd.DataFrame([{"split": k[0], "session": k[1], **rouse.file_summary(fh)} for k, fh in files.items()])
print("\nPer file:")
print(summary.drop(columns=["acquisition"]).to_string())
print("\nAcquisition groups in every file:", sorted(set(map(tuple, summary.acquisition))))

The only data streams are `preprocessed_emg` (16 muscles, unit mV, 20 ms steps), `eval_mask` and the
spike times of 64 multiunit channels; `processing`, `analysis` and `stimulus` are empty in every
file. There are no marker positions, joint angles or any other kinematic series, so every claim of
the paper about joint angles, marker trajectories or the forearm and elbow points is blocked. Every
file's subject is monkey L (identifier and subject id per file above); the dandiset holds no data of
monkeys X and Y. Its description says data of another monkey are in dandiset 001209 (FALCON M1-B,
monkey X, a draft with no published version); the claims file does not name it, so it is not used.

In [ ]:
print("Trial table columns, with their descriptions in the file:")
g = f["intervals/trials"]
for c in rouse.TRIAL_COLUMNS:
    print(f"  {c}: {g[c].attrs.get('description', '')}")

rows = []
for k, tt in trials.items():
    n = tt.number.to_numpy()
    rows.append({"split": k[0], "session": k[1], "trials": len(tt),
                 "results": dict(tt.result.value_counts()),
                 "number_min": n.min(), "number_max": n.max(),
                 "first_start_s": tt.start_time.min(), "last_stop_s": tt.stop_time.max()})
counts = pd.DataFrame(rows)
print("\nTrials per file:")
print(counts.to_string())

print("\nWhere trial number 127 sits in each file (neighbours in file order):")
for k, tt in trials.items():
    n = tt.number.to_numpy()
    for i in np.where(n == 127)[0]:
        print(f"  {k}: row {i}, numbers {n[max(0, i - 2):i + 3].tolist()}")

Every trial in every file has result `R` (rewarded): the files hold no failed trials, although the
column's description says it holds success, failure or abort. In the paper's session (held-in-calib,
20120928) the 412 trials carry numbers from 201 to 673, apart from one numbered 127, and span about
3,200 s. The paper's Table 2 gives this session 768 trials, 717 of them successful, over 1:26:11
(5,171 s). The trial with number 127 sits between 254 and 257 in the same session (and between 254
and 256 in two others), in time order, so its number is very likely 255 or 256, stored wrong; the
number 127 is the largest value of a signed 8-bit integer. Session 20120927 also has a true trial
127 at its start, so in that file the number does not identify a trial.

In [ ]:
print("Session 20120928, held-in-calib: numbers in file order, gaps between successive numbers")
n = t.number.to_numpy().astype(int)
n_fixed = n.copy()
n_fixed[(n == 127) & (np.arange(len(n)) > 0)] = -1
order_ok = np.all(np.diff(t.start_time.to_numpy()) > 0)
print("start times increase through the file:", order_ok)
valid = n_fixed[n_fixed > 0]
span = valid.max() - valid.min() + 1
print(f"numbers {valid.min()} to {valid.max()}: {span} numbers, {len(valid)} present besides the one numbered 127, "
      f"{span - len(valid) - 1} absent (counting 127 as one of 255 or 256)")
gaps = np.diff(valid)
print("absent numbers per gap:", pd.Series(gaps[gaps > 1] - 1).value_counts().sort_index().to_dict())

print("\nEvent times (s), session 20120928:")
ev = ["start_time", "gocue_time", "move_onset_time", "contact_time", "reward_time", "stop_time"]
print(t[ev].describe().round(3).to_string())
print("\nAll event times on a 20 ms grid:",
      bool(np.all(np.abs(np.round(t[ev].to_numpy() / 0.02) - t[ev].to_numpy() / 0.02) < 1e-6)))
d = pd.DataFrame({
    "cue minus start": t.gocue_time - t.start_time,
    "onset minus cue": t.move_onset_time - t.gocue_time,
    "contact minus onset": t.contact_time - t.move_onset_time,
    "reward minus contact": t.reward_time - t.contact_time,
    "stop minus reward": t.stop_time - t.reward_time})
print("\nIntervals between events (s):")
print(d.describe().round(3).to_string())

All event times lie on a 20 ms grid, the step of the EMG, so each is known only to 20 ms; the paper
times movement from 200 Hz video (5 ms) and contact from strain gauges sampled at 1 kHz. Each trial
starts exactly 1 s before the go cue and stops exactly 1 s after the reward, so `start_time` and
`stop_time` are a window cut around the events, not the trial's own start and stop. The reward time
comes 0 to 0.92 s after contact (median 0.10 s), where the paper's task requires a 1,000 ms hold of
the manipulated object before the water reward; `reward_time` therefore cannot be the time of the
paper's reward, and the file does not define it.
The events are in the expected order in every trial (all intervals positive).

In [ ]:
print("Objects: tgt_obj against obj_id, session 20120928")
print(pd.crosstab(t.tgt_obj, t.obj_id).to_string())
print("\nRuns of tgt_obj in file order:", [(k, len(list(g))) for k, g in itertools.groupby(t.tgt_obj)])

names, offsets = rouse.infer_object_names(t)
print("\nOffsets of each obj_id on the apparatus (most frequent difference of location from obj_id",
      offsets.obj_id.iloc[1], "between successive trials), and the names they imply:")
print(offsets.to_string(index=False))
t["object"] = t.obj_id.map(names)
t["orientation"] = t.tgt_loc - t.obj_id.map(dict(zip(offsets.obj_id, offsets.offset_deg)))
runs = [(k, len(list(g))) for k, g in itertools.groupby(t.orientation)]
print(f"\nApparatus orientation implied per trial (location minus offset): {len(runs)} runs in file order:")
print(runs)

print("\nTrials per object and location (columns: degrees; 157.5 is the leftmost location, 0 the rightmost):")
print(pd.crosstab(t.object, t.tgt_loc).to_string())
print("\ncondition_id is one to one with (obj_id, tgt_loc):",
      bool((t.groupby("condition_id")[["obj_id", "tgt_loc"]].nunique() == 1).all().all()
           and t.groupby(["obj_id", "tgt_loc"]).condition_id.nunique().eq(1).all()))

`tgt_obj` ("object to grasp at target") takes three values (Sphere, Button, Coaxial) in three long
runs of 118, 159 and 135 trials, and each value occurs with all four `obj_id` codes and with every
location. It cannot be the object grasped: the paper's monkeys grasped four objects, presented in
blocks at each apparatus orientation. `obj_id` ("integer id for each object") has four codes, and
within each run of trials the location minus a fixed offset per code is constant, with offsets 45
degrees apart, as the paper's apparatus has them; the orientations implied fall in 15 runs of 12 to
40 trials, consistent with the paper's blocks at one apparatus orientation. Named in the clockwise
order of the Fig. 1 legend (perpendicular cylinder, coaxial cylinder, button, sphere, with 157.5
degrees leftmost), the codes are 2, 1, 4 and 3. The names agree with every constraint the legend
states: the coaxial cylinder is at all eight locations (the legend's frames show it at eight), the
sphere and button are never at the leftmost locations, and the perpendicular cylinder is not at the
rightmost ones; 4 + 6 + 6 + 8 = 24 combinations, as in the paper. The names are inferred from the
paper's legend and the structure of the trials; the files do not name the codes. `condition_id`
is one to one with object and location. The EMG check below supports `obj_id` as the grasped object.

In [ ]:
print("Recordings deposited more than once: held-in-minival against held-in-calib, per session")
for s in sorted({k[1] for k in trials if k[0] == "held-in-minival"}):
    mv, ca = trials[("held-in-minival", s)], trials[("held-in-calib", s)]
    same = ca.merge(mv, on=["number", "start_time"], suffixes=("_c", "_m"))
    cols_equal = all(np.allclose(same[c + "_c"], same[c + "_m"]) for c in ["gocue_time", "move_onset_time", "contact_time", "obj_id", "tgt_loc"])
    ts_m, emg_m, _ = rouse.read_emg(files[("held-in-minival", s)])
    gc = files[("held-in-calib", s)]["acquisition/preprocessed_emg"]
    names_m = sorted(gc.keys())
    emg_c = np.column_stack([gc[nm]["data"][:len(ts_m)] for nm in names_m])
    ts_c = gc[names_m[0]]["timestamps"][:len(ts_m)]
    st_m = rouse.read_spike_times(files[("held-in-minival", s)])
    st_c = rouse.read_spike_times(files[("held-in-calib", s)])
    n_same = sum(np.array_equal(a, b[b <= (a.max() if len(a) else -1)]) for a, b in zip(st_m, st_c))
    ratio = emg_m / emg_c
    r = np.corrcoef(emg_m.ravel(), emg_c.ravel())[0, 1]
    print(f"  {s}: minival trials {mv.number.tolist()} found in calib: {len(same)}; event times equal: {cols_equal}; "
          f"EMG timestamps equal: {np.array_equal(ts_m, ts_c)}, values identical: {np.array_equal(emg_m, emg_c)}, "
          f"correlation {r:.5f}, median ratio minival/calib per muscle {np.nanmin(np.nanmedian(ratio, 0)):.4f} to "
          f"{np.nanmax(np.nanmedian(ratio, 0)):.4f}; channels with identical spike times: {n_same} of {len(st_m)}")
print("\nSessions in held-out-calib:", sorted(k[1] for k in trials if k[0] == "held-out-calib"),
      "(none is the paper's session)")

Each `held-in-minival` file repeats the first two trials of the `held-in-calib` file of the same
session, with the same event times and the same EMG timestamps; the spike times and EMG agree as
printed above (the EMG of the two files differs by a near-constant scale factor, so each file's EMG
was normalized on its own). They are the same recording deposited twice, as the dataset's
description says (minival is a subset of calib). Only the `held-in-calib` file of
20120928 is used below, and each trial is counted once.

In [ ]:
ts, emg, muscles = rouse.read_emg(f)
spikes = rouse.read_spike_times(f)
print(f"EMG: {emg.shape[1]} muscles {muscles}, {emg.shape[0]} samples, step {np.median(np.diff(ts)) * 1000:.0f} ms, "
      f"{ts[0]:.2f} to {ts[-1]:.2f} s, values {np.nanmin(emg):.3f} to {np.nanmax(emg):.3f} (unit mV in the file), "
      f"NaN fraction {np.isnan(emg).mean():.4f}")
n_sp = np.array([len(s) for s in spikes])
print(f"Spikes: {len(spikes)} channels, {n_sp.sum()} spikes, per channel {n_sp.min()} to {n_sp.max()}, "
      f"times {min(s.min() for s in spikes if len(s)):.2f} to {max(s.max() for s in spikes if len(s)):.2f} s")

# Clock check: decode the grasped object (obj_id) from activity in 100 ms windows at lags from contact.
# With a shared clock, accuracy should be near chance well before the cue and high around contact.
# Labels from the next trial (shifted by one trial) are the control.
y = t.obj_id.to_numpy()
y_shift = np.roll(y, -1)
rows = []
for lag in np.round(np.arange(-1.6, 0.61, 0.1), 2):
    Xe = rouse.window_means(ts, emg, t.contact_time.to_numpy(), lag, lag + 0.1)
    Xs = rouse.spike_counts(spikes, t.contact_time.to_numpy(), lag, lag + 0.1)
    rows.append({"lag_from_contact_s": lag,
                 "emg_obj_id": rouse.decode_object(Xe, y), "emg_obj_id_next_trial": rouse.decode_object(Xe, y_shift),
                 "emg_tgt_obj": rouse.decode_object(Xe, t.tgt_obj.to_numpy()),
                 "spikes_obj_id": rouse.decode_object(Xs, y)})
clock = pd.DataFrame(rows)
chance = t.obj_id.value_counts(normalize=True).max()
print(f"\nCross-validated accuracy of decoding (chance as the largest class: obj_id {chance:.3f}, "
      f"tgt_obj {t.tgt_obj.value_counts(normalize=True).max():.3f}); median cue to contact "
      f"{(t.contact_time - t.gocue_time).median():.2f} s:")
print(clock.round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for c, lab in [("emg_obj_id", "EMG, obj_id"), ("spikes_obj_id", "spikes, obj_id"),
               ("emg_obj_id_next_trial", "EMG, next trial's obj_id"), ("emg_tgt_obj", "EMG, tgt_obj")]:
    ax.plot(clock.lag_from_contact_s + 0.05, clock[c], marker="o", ms=3, label=lab)
ax.axhline(chance, color="grey", ls=":", lw=1)
ax.axvline(0, color="k", lw=0.5)
ax.axvline(-(t.contact_time - t.gocue_time).median(), color="k", lw=0.5, ls="--")
ax.set_xlabel("window centre from contact (s); dashed: median cue")
ax.set_ylabel("decoding accuracy")
ax.set_ylim(0, 1.02)
ax.legend(fontsize=7)
print("All plotted values lie within the axes:", float(clock.iloc[:, 1:].min().min()), "to", float(clock.iloc[:, 1:].max().max()))

The EMG and the spikes share the clock of the trial table: the object (`obj_id`) is decoded near
chance (0.26 to 0.35, against 0.345 for the largest class) from EMG at lags of a second or more
before contact, before the median go cue (0.5 s before contact), rises from about 0.7 s before
contact, and is decoded at 0.95 to 0.99 from 0.2 s before contact to 0.1 s after it (spikes 0.84 to
0.90), while the next trial's object is decoded at 0.40 to 0.45 at those lags. `tgt_obj` is decoded no better
around contact than a second before it, so it does not label what the hand did; this agrees with
reading `obj_id` as the grasped object. Only the trial table enters the claims below; the event
times themselves (onset and contact) could not be checked against kinematics, as the dataset has
none. The movement onset and contact times are consistent with the EMG (the object is decoded from
EMG in the window just before contact), which is consistent with, but does not show, the paper's
definitions.

## table2-trial-counts

> In the three sessions the monkeys performed 768, 824 and 745 trials (L, X, Y), of which 717, 788
> and 688 were successful and 685, 752 and 637 were analyzed.

Criterion: counts per session from the trial records; analyzed trials are successful trials not
preceded by an error trial. Monkeys X and Y are not in the dataset. For monkey L the counts are
taken from the `held-in-calib` file of session 20120928 (the minival file repeats two of its trials).

In [ ]:
n_success = int((t.result == "R").sum())
n_fail = int((t.result != "R").sum())
print(f"Session 20120928: {len(t)} trials in the file, {n_success} rewarded, {n_fail} failed or aborted")
print(f"Paper (Table 2): 768 total (tolerance 691 to 845), 717 successful (645 to 789), 685 analyzed (617 to 754)")
print(f"File spans {t.start_time.min():.0f} to {t.stop_time.max():.0f} s ({(t.stop_time.max() - t.start_time.min()) / 60:.1f} min); "
      f"paper: session of 1:26:11 ({5171 / 60:.1f} min)")
print(f"Trial numbers {valid.min()} to {valid.max()}; numbers below {valid.min()} and above {valid.max()} are absent; "
      f"{span - len(valid) - 1} numbers inside the span are absent")
prev_present = np.isin(n_fixed - 1, n_fixed[n_fixed > 0]) & (n_fixed > 0)
print(f"Variant, if every absent number inside the span were an error trial: "
      f"total in the span {span}, successful {n_success}, successful with the preceding number present {int(prev_present.sum())}")

**Result: blocked, data missing.** The file of session L20120928 holds 412 rewarded trials and no
failed ones, numbered 201 to 673 and spanning about 53 of the session's 86 minutes, against 768
trials and 717 successes in the paper. The dataset holds only the part of the session released for
the FALCON benchmark's calibration split, so none of the nine counts can be taken from it, and X and
Y are not in it. The difference comes from the selection, not from the paper.

## table2-movement-time

> The median duration from movement onset to contact differs among the monkeys (255, 235 and 335 ms
> for L, X and Y; Kruskal-Wallis P < 0.001), giving 52, 48 and 68 normalized samples from M to C.

Criterion: movement time from onset to contact over the analyzed trials (a) or all successful
trials (b); the claims file says that onset times given by the dataset are used. The dataset gives
`move_onset_time` and `contact_time`, so the marker-speed onset is not recomputed (and could not be:
there are no markers). The number of samples from M to C, inclusive of both, at 200 Hz is
median / 5 ms + 1 (Results, paragraph 1: "inclusive of the first sample at movement onset and the
last sample at peripheral object contact"; 255 ms gives 52). Only monkey L is in the dataset.
Reading (b) uses the 412 successful trials in the file; reading (a) cannot be formed exactly, as the
error trials are absent, and is approximated by the trials whose preceding trial number is present.

In [ ]:
mt = (t.contact_time - t.move_onset_time) * 1000
readings = {"(b) successful trials in the file": mt,
            "(a) approx.: preceding number present": mt[prev_present]}
rows = []
for name, v in readings.items():
    q = v.quantile([0.25, 0.5, 0.75])
    rows.append({"reading": name, "n": len(v), "median_ms": q[0.5], "q25_ms": q[0.25], "q75_ms": q[0.75],
                 "samples_M_to_C": q[0.5] / (1000 / rouse.SAMPLE_HZ) + 1,
                 "median_within_229.5_280.5": 229.5 <= q[0.5] <= 280.5})
res = pd.DataFrame(rows)
print("Monkey L, session 20120928. Paper: median 255 ms (225, 295), 52 samples; tolerance 229.5 to 280.5 ms, 46.8 to 57.2 samples")
print(res.round(1).to_string(index=False))
print(f"\nMovement times range {mt.min():.0f} to {mt.max():.0f} ms; 20 ms steps, so each value is known to +-20 ms")
print("Fraction of trials with movement time >= 255 ms:", round(float((mt >= 255).mean()), 3))

print("\nMedian movement time by object (names inferred above) and by location, ms:")
print(t.assign(mt=mt).groupby("object").mt.agg(["median", "size"]).to_string())
print(t.assign(mt=mt).groupby("tgt_loc").mt.median().to_string())
half = t.start_time < t.start_time.median()
print(f"\nFirst half of the file: median {mt[half].median():.0f} ms; second half: {mt[~half].median():.0f} ms")

other = {s: ((trials[('held-in-calib', s)].contact_time - trials[('held-in-calib', s)].move_onset_time) * 1000).median()
         for s in sorted(k[1] for k in trials if k[0] == "held-in-calib")}
print("For reference, median movement time in the other monkey L sessions of the dataset (ms):", other)
print("\nKruskal-Wallis across monkeys: not run; monkeys X and Y are not in the dataset.")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
bins = np.arange(90, 760, 20)
ax.hist(mt, bins=bins, color="0.5")
ax.axvline(mt.median(), color="C0", label=f"median here {mt.median():.0f} ms")
ax.axvspan(229.5, 280.5, color="C1", alpha=0.25, label="paper 255 ms, tolerance")
ax.set_xlabel("contact minus move onset (ms)")
ax.set_ylabel("trials")
ax.legend(fontsize=7)
print(f"Plotted range {mt.min():.0f} to {mt.max():.0f} ms, all within the axes")

**Result: not reproduced.** The median time from the dataset's movement onset to contact in session
L20120928 is 200 ms (quartiles 160 and 240 ms) over the 412 successful trials in the file, and
200 ms over the 387 trials whose preceding trial number is present. The paper's 255 ms (225, 295) lies
outside the tolerance of 229.5 to 280.5 ms, and the implied 41 samples from M to C are outside 47
to 57. The 20 ms steps of the event times cannot account for a 55 ms difference of the median. The
dataset does not describe how `move_onset_time` was found; a different definition of onset than the
paper's marker-speed threshold would explain the difference, and the trials in the file are a part
of the session's. The comparison among monkeys and its Kruskal-Wallis test cannot be made, as X and
Y are not in the dataset.

## fig2-trial-count-coax-45

> Monkey L successfully reached to, grasped and manipulated the coaxial cylinder at the 45 degree
> location in 28 trials of its session.

Criterion: analyzed (a) or all successful (b) trials of session L20120928 with that object and
location. The object is named from `obj_id` as in the data check (the files' `tgt_obj` is not the
object).

In [ ]:
sel = (t.object == "coaxial cylinder") & (t.tgt_loc == 45.0)
print(f"Coaxial cylinder (obj_id {[k for k, v in names.items() if v == 'coaxial cylinder'][0]}) at 45 degrees: "
      f"(b) {int(sel.sum())} successful trials in the file, (a) approx. {int((sel & prev_present).sum())} with the preceding number present")
print("Paper: 28 (tolerance 25.2 to 30.8)")
print("With the file's tgt_obj == 'Coaxial' at 45 degrees:", int(((t.tgt_obj == "Coaxial") & (t.tgt_loc == 45.0)).sum()))
print(f"Successful trials of the session in the file: {len(t)} of the paper's 717 ({len(t) / 717:.2f}); "
      f"28 x {len(t) / 717:.2f} = {28 * len(t) / 717:.1f}")

**Result: blocked, data missing.** The file holds 19 successful trials of the coaxial cylinder at 45
degrees (16 with the preceding trial number present), against 28 in the paper; it holds 412 of the 717
successful trials of the session (57%), and 57% of 28 is 16. The count cannot be taken from a part
of the session.

## Claims blocked for missing kinematic data

The other 23 claims (fig2-joint-patterns, fig2-hand-opening-two-phases, fig4-armps-time-course,
fig5-location-proximal-object-distal, fig5-early-significance-concurrent, fig5-shoulderier-location,
fig5-shoulder-object-effects, fig5-elbowflex-object, fig5-wrist-comparable,
fig5-hand-object-predominates, fig5-location-early-object-late, fig5-interaction-small,
fig6a-qm-30-90, fig6b-qx-small, fig7-mean-eta-30, fig7-mean-eta-90, fig7-time-course,
fig8a-lda-location, fig8b-lda-object, fig8-hand-shape-invariant-to-location,
fig9-hand-configuration-location, fig10-forearm-trajectory-object,
elbow-point-trajectory-object) are all tests on the 22 joint angles or on marker positions, in
three monkeys. The data check shows that no file holds a marker position, a joint angle or any
other kinematic series, and that only monkey L is in the dataset. They are blocked, data missing.

In [ ]:
import h5py
kin = []
for k, fh in files.items():
    def look(name, obj):
        if isinstance(obj, h5py.Dataset) and not name.startswith("specifications"):
            if name.split("/")[0] in ("acquisition", "processing", "analysis", "stimulus"):
                kin.append((k, name))
    fh.visititems(look)
streams = sorted({n.split("/")[0] + "/" + n.split("/")[1] for _, n in kin})
print("Data series outside the trial table and units, over all 11 files:", streams)
print(f"Done in {time.time() - started:.0f} s")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| table2-trial-counts | blocked: data missing |
| table2-movement-time | not reproduced (monkey L only) |
| fig2-trial-count-coax-45 | blocked: data missing |
| the other 23 claims (joint angles and marker positions) | blocked: data missing |

**How the analysis differs from the paper's.** Movement onset and contact are the dataset's own
event times, at 20 ms resolution, not the paper's onset from marker speed (5 ms steps) and contact
from strain gauges (1 kHz). Only monkey L is in the dataset, and only 412 of the 717 successful
trials of its session. The objects are named from `obj_id` by the apparatus order of the paper's
Fig. 1 legend.

**What was found about the dataset.** It holds trial tables, EMG and multiunit spikes of monkey L,
and no kinematics, so it cannot support the paper it names first. The session L20120928 is there
only in part (trials 201 to 673, about 53 of 86 minutes, no failed trials). `tgt_obj` does not name
the grasped object; `obj_id` does, without names. Trial number 255 or 256 is stored as 127 in each
of the four calibration files. The minival files repeat the first two trials of the calibration
files. `move_onset_time`, `contact_time` and `reward_time` are not defined, and `reward_time` comes a median 0.10 s after contact, where the paper's task requires a 1 s hold
before the reward.